# Single Question Evaluation Tester

Use this notebook to test the generation and evaluation pipeline for a single question. Adjust the parameters in the configuration cell below.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
import os
import json
from dotenv import load_dotenv

# Ensure the project root is in python path if not already
if os.path.abspath(".") not in sys.path:
    sys.path.append(os.path.abspath("."))

from parrot_ai.llm_evaluation import EvaluationEngine
from parrot_ai.prompts.english import MAIN_SYSTEM_PROMPT

load_dotenv()
print("Environment loaded.")

/Users/spaceairmac/.virtualenvs/cp_multilingual/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment loaded.


In [ ]:
# --- Configuration ---

# The Question to test
# QUESTION = "What is the significance of Mount Hermon in the Bible?"
# QUESTION = "Who was Herod Agrippa I?" # Test factual
# QUESTION = "Is hell real?" # Test controversial
# QUESTION = "Why do Christians keep saying that I'm going to hell? I'm a good person!"
QUESTION = "Who is the ancient serpent of Revelation 12:9?" # Test theological / interpretive question

# Generation Parameters
PROVIDER = "google"     # options: openai, google, anthropic, xai, together, hf
GEN_MODEL = "gemini-3-flash-preview" 
USE_SYSTEM_PROMPT = True

# Evaluation Parameters
JUDGE_MODEL = "gpt-5-mini"  # The model used to score the answer (defaults to gpt-5-mini in code)
LANGUAGE = "english"

In [3]:
# --- Initialize Engine ---
# Note: The engine uses the judge model for evaluation.
engine = EvaluationEngine(model=JUDGE_MODEL, language=LANGUAGE)
print(f"EvaluationEngine initialized with judge model: {JUDGE_MODEL}")

EvaluationEngine initialized with judge model: gpt-5-mini


In [4]:
# --- Generate Answer ---
system_prompt = MAIN_SYSTEM_PROMPT if USE_SYSTEM_PROMPT else None

if system_prompt:
    print("Using system prompt for generation.")

print(f"Generating answer using {PROVIDER}:{GEN_MODEL}...")

# 'generate_responses' handles the provider API calls internally
responses = engine.generate_responses(
    questions=[QUESTION],
    provider=PROVIDER,
    model=GEN_MODEL,
    system=system_prompt
)

if responses and "answer" in responses[0]:
    generated_answer = responses[0]["answer"]
    print("\n" + "="*40)
    print("GENERATED ANSWER:")
    print("-" * 40)
    print(generated_answer)
    print("="*40)
else:
    print("Error generating response:", responses)

Using system prompt for generation.
Generating answer using google:gemini-3-flash-preview...
Gemini client initialized
Model set to: gemini-3-flash-preview


Generating (Google): 100%|██████████| 1/1 [00:10<00:00, 10.54s/q]


GENERATED ANSWER:
----------------------------------------
The ancient serpent mentioned in Revelation 12:9 is explicitly identified within the same verse as the devil and Satan. The text describes him as the great dragon and the deceiver of the whole world who was thrown down to the earth (Revelation 12:9). This title "ancient serpent" points back to the beginning of human history in the book of Genesis.

In the garden of Eden, the serpent appeared as a crafty creature that twisted God's words and deceived Adam and Eve into rebelling against their Creator (Genesis 3:1–7). This event brought sin, suffering, and death into the world, separating humanity from a holy and perfect God. However, even in the moment of judgment, God promised that a descendant of the woman would one day come to crush the serpent's head (Genesis 3:15).

That promise was fulfilled in Jesus Christ. While the serpent seeks to steal, kill, and destroy, Jesus came so that we might have life. Through His perfect life

In [5]:
# --- Evaluate Answer ---
print(f"Evaluating answer with Judge ({JUDGE_MODEL})...")

# 'evaluate' runs the rubric scoring steps
eval_result = engine.evaluate(QUESTION, generated_answer)

# Display Result as pretty JSON
print(json.dumps(eval_result, indent=2, ensure_ascii=False))

Evaluating answer with Judge (gpt-5-mini)...
{
  "Adherence": {
    "Core": 5,
    "Secondary": 5,
    "Tertiary_Handling": 5,
    "Biblical_Basis": 5,
    "Consistency": 5,
    "Overall": 5
  },
  "Kindness_and_Gentleness": {
    "Core_Clarity_with_Kindness": 5,
    "Pastoral_Sensitivity": 3,
    "Secondary_Fairness": 5,
    "Tertiary_Neutrality": 5,
    "Tone": 4,
    "Overall": 5
  },
  "Interfaith_Sensitivity": {
    "Respect_and_Handling_Objections": 5,
    "Objection_Acknowledgement": 5,
    "Evangelism": 5,
    "Gospel_Boldness": 5,
    "Overall": 5
  },
  "Arabic_Accuracy": {}
}
